# Dataset Exploration
EDA notebook for the synthetic dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

def load_data():
    aws = pd.read_parquet('../output/aws/aws_billing_labeled.parquet')
    azure = pd.read_parquet('../output/azure/azure_billing_labeled.parquet')
    gcp = pd.read_parquet('../output/gcp/gcp_billing_labeled.parquet')
    return aws, azure, gcp
try:
    aws, azure, gcp = load_data()
    combined = pd.concat([aws, azure, gcp])
except Exception as e:
    print('Generate data first using generate.py')

In [ ]:
# 1. Daily total cost per cloud — line chart
if 'combined' in locals():
    combined['usage_date_dt'] = pd.to_datetime(combined['usage_date'])
    daily_costs = combined.groupby(['usage_date_dt', 'cloud_provider'])['cost_usd'].sum().unstack()
    daily_costs.plot(figsize=(15, 6), title='Daily Total Cost per Cloud')
    plt.show()

In [ ]:
# 2. Anomaly overlay
if 'combined' in locals():
    ax = daily_costs.plot(figsize=(15, 6), title='Daily Total Cost with Anomalies', alpha=0.5)
    anomalies = combined[combined['is_anomaly'] == True]['usage_date_dt'].unique()
    for a_date in anomalies:
        ax.axvline(a_date, color='red', alpha=0.1)
    plt.show()

In [ ]:
# 3. Cost distribution by service — box plot per cloud
if 'combined' in locals():
    for c in ['aws', 'azure', 'gcp']:
        df_c = combined[combined['cloud_provider'] == c]
        df_c.boxplot(column='cost_usd', by='service', figsize=(15,6), rot=45)
        plt.title(f'Cost Distribution by Service - {c.upper()}')
        plt.suptitle('')
        plt.show()

In [ ]:
# 4. Anomaly type breakdown — horizontal bar chart
if 'combined' in locals():
    anom_counts = combined[combined['is_anomaly'] == True]['anomaly_type'].value_counts()
    anom_counts.plot(kind='barh', title='Anomaly Type Breakdown')
    plt.show()

In [ ]:
# 5. Tag key frequency — top 20 tags across all records
if 'combined' in locals():
    # flatten dicts
    tag_keys = [k for tags in combined['tags'].dropna() if isinstance(tags, dict) for k in tags.keys()]
    pd.Series(tag_keys).value_counts().head(20).plot(kind='bar', title='Top 20 Tag Keys')
    plt.show()

In [ ]:
# 6. Edge case summary — pie chart of edge case types
if 'combined' in locals():
    dups = combined['is_duplicate'].sum()
    backs = combined['is_backdated'].sum()
    negs = (combined['cost_usd'] < 0).sum()
    pd.Series({'Duplicates': dups, 'Backdated': backs, 'Negative Cost': negs}).plot(kind='pie', title='Edge Cases')
    plt.show()

In [ ]:
# 7. Monthly spend vs budget — bar chart with budget threshold line
if 'combined' in locals():
    monthly = combined.set_index('usage_date_dt').groupby('cloud_provider').resample('M')['cost_usd'].sum().unstack(0)
    budgets = {'aws': 180000, 'azure': 160000, 'gcp': 140000}
    for c in ['aws', 'azure', 'gcp']:
        monthly[c].plot(kind='bar', title=f'{c.upper()} Monthly Spend', color='lightblue')
        plt.axhline(budgets[c], color='red', linestyle='--', label=f'{c.upper()} Budget')
        plt.legend()
        plt.show()

In [ ]:
# 8. Currency distribution — stacked area chart showing original_currency over time
if 'combined' in locals():
    currency_counts = combined.groupby(['usage_date_dt', 'original_currency']).size().unstack().fillna(0)
    currency_counts.plot(kind='area', stacked=True, figsize=(15, 6), title='Currency Distribution Over Time')
    plt.show()